In [12]:
import json
import os

In [13]:
def evaluate_data(data):
    operator_counts = {}
    ic_count = 0
    ec_count = 0

    def recurse_entries(section_data):
        nonlocal ic_count, ec_count
        if isinstance(section_data, dict):
            for key, value in section_data.items():
                if key.startswith('IC'):
                    ic_count += 1
                if key.startswith('EC'):
                    ec_count += 1
                if key == 'operators' and isinstance(value, list):
                    for operator in value:
                        operator_counts[operator] = operator_counts.get(operator, 0) + 1
                # Rekursiv alle Unterelemente durchsuchen
                if isinstance(value, dict):
                    recurse_entries(value)
                elif isinstance(value, list):
                    for item in value:
                        if isinstance(item, dict):
                            recurse_entries(item)

    # Starte die rekursive Suche in IC und EC
    if 'IC' in data:
        recurse_entries(data['IC'])
    if 'EC' in data:
        recurse_entries(data['EC'])

    return {
        "ic_count": ic_count,
        "ec_count": ec_count,
        "operator_counts": operator_counts
    }

def read_and_evaluate_files(input_directory, output_directory):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    for filename in os.listdir(input_directory):
        if filename.endswith(".json"):
            filepath = os.path.join(input_directory, filename)
            with open(filepath, 'r', encoding='utf-8') as file:
                data = json.load(file)
                results = evaluate_data(data)
                output_filepath = os.path.join(output_directory, filename.replace(".json", "_eval.json"))
                with open(output_filepath, 'w', encoding='utf-8') as output_file:
                    json.dump(results, output_file, indent=4)

In [14]:
input_directory = '../output/struct'
label_directory = 'chia_eval'
llama3_directory = 'n_shot_llama3_inst'

read_and_evaluate_files(input_directory, output_directory)